## 02 — Wrangle Data
Hàm `wrangle()` tải dữ liệu và xây dựng đặc trưng RFM (Recency, Frequency, Monetary) cho từng đại lý.

> ~24% đơn không gắn customer_code (đơn lẻ) — loại bỏ trước khi tính RFM.

In [ ]:
def wrangle(filepath):
    df = pd.read_csv(filepath, parse_dates=["order_date"])
    # Loại bỏ đơn không có thông tin đại lý
    df = df.dropna(subset=["customer_code", "customer_name"]).copy()
    df["line_total"] = df["line_total"].fillna(0)
    return df

df = wrangle("data/fact_sales_full.csv")
print("df shape:", df.shape)
print("Số đại lý:", df.customer_code.nunique())
df.head()

In [ ]:
# Xây dựng đặc trưng RFM
reference_date = df["order_date"].max()

rfm = df.groupby("customer_code").agg(
    recency   = ("order_date", lambda x: (reference_date - x.max()).days),
    frequency = ("so_number",  "nunique"),
    monetary  = ("line_total", "sum"),
).reset_index()

# Đặc trưng bổ sung: số đơn 3 tháng gần nhất
cutoff_3m = reference_date - pd.DateOffset(months=3)
last3m = (
    df[df["order_date"] >= cutoff_3m]
    .groupby("customer_code")["so_number"]
    .nunique()
    .reset_index()
    .rename(columns={"so_number": "last3m_orders"})
)
rfm = rfm.merge(last3m, on="customer_code", how="left")
rfm["last3m_orders"] = rfm["last3m_orders"].fillna(0).astype(int)

print("rfm shape:", rfm.shape)
rfm.head()

In [ ]:
# Gán nhãn Churn: recency > 90 ngày = churn
CHURN_THRESHOLD = 90
rfm["churn"] = (rfm["recency"] > CHURN_THRESHOLD).astype(int)

print(f"Ngưỡng churn : recency > {CHURN_THRESHOLD} ngày")
print(f"Churn        : {rfm.churn.sum()} / {len(rfm)}  ({rfm.churn.mean()*100:.1f}%)")